AQUI EMPEZAMOS CON LA CREACION DEL CHAT BOT DE MINERALES...


In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime

# Minerales e hidrocarburos que vamos a trackear
tickers = {
    "Cobre":      "HG=F",
    "Oro":        "GC=F",
    "Plata":      "SI=F",
    "Zinc":       "ZNC=F",
    "WTI":        "CL=F",
    "Brent":      "BZ=F"
}

# Extraer precios de los últimos 7 días
datos = []

for nombre, ticker in tickers.items():
    activo = yf.Ticker(ticker)
    hist = activo.history(period="7d")
    
    if not hist.empty:
        ultimo_precio = hist["Close"].iloc[-1]
        precio_anterior = hist["Close"].iloc[-2] if len(hist) > 1 else ultimo_precio
        variacion = ((ultimo_precio - precio_anterior) / precio_anterior) * 100
        
        datos.append({
            "mineral": nombre,
            "precio": round(ultimo_precio, 2),
            "variacion_pct": round(variacion, 2),
            "fecha": datetime.now().strftime("%Y-%m-%d %H:%M")
        })
        print(f"✅ {nombre}: ${ultimo_precio:.2f} ({variacion:+.2f}%)")
    else:
        print(f"⚠️ {nombre}: sin datos")

df_precios = pd.DataFrame(datos)
print("\n--- DataFrame final ---")
print(df_precios)

✅ Cobre: $5.99 (+3.33%)
✅ Oro: $4570.00 (+1.12%)
✅ Plata: $73.65 (+0.79%)
✅ Zinc: $2297.00 (+0.00%)
✅ WTI: $102.27 (-3.90%)
✅ Brent: $109.88 (-3.98%)

--- DataFrame final ---
  mineral   precio  variacion_pct             fecha
0   Cobre     5.99           3.33  2026-05-05 13:44
1     Oro  4570.00           1.12  2026-05-05 13:44
2   Plata    73.65           0.79  2026-05-05 13:44
3    Zinc  2297.00           0.00  2026-05-05 13:44
4     WTI   102.27          -3.90  2026-05-05 13:44
5   Brent   109.88          -3.98  2026-05-05 13:44


SEGUNDO PASO PARA HACER EL CHAT BOT DE MINERALES


In [2]:
import psycopg2
from psycopg2.extras import execute_values

# Conexión a tu PostgreSQL en Docker
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="minerales_bot",
    user="admin",
    password="admin123"
)
cursor = conn.cursor()

# Crear tabla si no existe
cursor.execute("""
    CREATE TABLE IF NOT EXISTS precios_minerales (
        id SERIAL PRIMARY KEY,
        mineral VARCHAR(50),
        precio NUMERIC(10,2),
        variacion_pct NUMERIC(5,2),
        fecha TIMESTAMP DEFAULT NOW()
    )
""")

conn.commit()
print("✅ Tabla creada correctamente")
cursor.close()
conn.close()

OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?


3ER PASO 

In [ ]:
# Conectar e insertar los precios
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="minerales_bot",
    user="admin",
    password="admin123"
)
cursor = conn.cursor()

# Insertar cada fila del DataFrame
for _, fila in df_precios.iterrows():
    cursor.execute("""
        INSERT INTO precios_minerales (mineral, precio, variacion_pct, fecha)
        VALUES (%s, %s, %s, %s)
    """, (fila["mineral"], fila["precio"], fila["variacion_pct"], fila["fecha"]))

conn.commit()
print(f"✅ {len(df_precios)} precios insertados en PostgreSQL")

# Verificar que llegaron bien
cursor.execute("SELECT * FROM precios_minerales ORDER BY id DESC LIMIT 6")
rows = cursor.fetchall()
print("\n--- Datos en la base de datos ---")
for row in rows:
    print(row)

cursor.close()
conn.close()

✅ 6 precios insertados en PostgreSQL

--- Datos en la base de datos ---
(6, 'Brent', Decimal('111.22'), Decimal('-2.81'), datetime.datetime(2026, 5, 5, 9, 45))
(5, 'WTI', Decimal('102.20'), Decimal('-3.97'), datetime.datetime(2026, 5, 5, 9, 45))
(4, 'Zinc', Decimal('2297.00'), Decimal('0.00'), datetime.datetime(2026, 5, 5, 9, 45))
(3, 'Plata', Decimal('74.11'), Decimal('1.43'), datetime.datetime(2026, 5, 5, 9, 45))
(2, 'Oro', Decimal('4591.30'), Decimal('1.59'), datetime.datetime(2026, 5, 5, 9, 45))
(1, 'Cobre', Decimal('6.01'), Decimal('3.66'), datetime.datetime(2026, 5, 5, 9, 45))


In [10]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Configuración — pon tu correo Gmail aquí
CORREO_ORIGEN = "tucorreo@gmail.com"
CORREO_DESTINO = "correo_del_cliente@gmail.com"
PASSWORD_APP = "aqui_va_tu_password_de_app"

# Umbrales de alerta por mineral
UMBRALES = {
    "Cobre": {"min": 5.50, "max": 5.98},  
    "Oro":   {"min": 4000, "max": 5000},
    "WTI":   {"min": 80,   "max": 110}
}

def generar_alertas(df):
    alertas = []
    for _, fila in df.iterrows():
        mineral = fila["mineral"]
        precio = fila["precio"]
        if mineral in UMBRALES:
            if precio >= UMBRALES[mineral]["max"]:
                alertas.append(f"🔴 ALERTA ALTA: {mineral} llegó a ${precio} — por encima del umbral máximo de ${UMBRALES[mineral]['max']}")
            elif precio <= UMBRALES[mineral]["min"]:
                alertas.append(f"🟢 ALERTA BAJA: {mineral} cayó a ${precio} — por debajo del umbral mínimo de ${UMBRALES[mineral]['min']}")
    return alertas

alertas = generar_alertas(df_precios)

if alertas:
    for a in alertas:
        print(a)
else:
    print("✅ Sin alertas por ahora — precios dentro de rangos normales")

🔴 ALERTA ALTA: Cobre llegó a $5.99 — por encima del umbral máximo de $5.98
